In [ ]:
import os
import time
import threading
import concurrent.futures
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm

"""inchikey-massbank_v4.py

- Preserve and enhance Ion Mode / LC chromatographic parameter extraction with compatible column names
- Added: When Collision Energy is not directly provided in the record, automatically fetch record details
  via MassBank API /records/{accession} endpoint to extract Collision Energy.

Note:
- This script uses /records?inchi_key=... to retrieve candidate record lists
- For each record, if Collision Energy is missing (or mode is 'always'), it will additionally call
  /records/{accession} to fetch details (with caching) and extract Collision Energy
"""

# -------------------- Configuration --------------------

API_ROOT = "https://massbank.eu/MassBank-api"
API_RECORDS_URL = f"{API_ROOT}/records"
API_RECORD_URL_TMPL = f"{API_ROOT}/records/{{accession}}"

INPUT_FILE = "汇总8749.xlsx"
OUTPUT_FILE = "8749-特征.csv"

MAX_WORKERS = 5          # Number of concurrent threads
RETRY_COUNT = 3          # Number of retries on failure
BATCH_SIZE = 100         # Write to file every 100 future results

# Global rate limiting (to avoid overwhelming the API), unit: seconds/request
# Set to 0 for no rate limiting
REQUEST_INTERVAL_SECONDS = 0

# Collision Energy fetch strategy:
# - "if_missing": Only request /records/{accession} when Collision Energy cannot be parsed from current record
# - "always":     Request /records/{accession} for every record (slower but more reliable)
COLLISION_ENERGY_FETCH_MODE = "if_missing"


# -------------------- Utility Functions --------------------

def get_processed_inchikeys(output_file: str) -> set:
    """Read already processed InChIKeys from output file for resuming interrupted runs."""
    if not os.path.exists(output_file):
        return set()
    try:
        df = pd.read_csv(output_file, on_bad_lines="warn")
        if "InChIKey" in df.columns:
            return set(df["InChIKey"].dropna().unique())
        return set()
    except (pd.errors.EmptyDataError, FileNotFoundError):
        return set()


def extract_nested_value(data, key_path, default="N/A"):
    """Safely extract a value from a nested dictionary."""
    try:
        for key in key_path:
            data = data[key]
        return data
    except (KeyError, TypeError, IndexError):
        return default


def find_subtag_value(subtags, subtag_name, default="N/A"):
    """Find a specific subtag value from subtags list."""
    if not isinstance(subtags, list):
        return default
    for item in subtags:
        if item.get("subtag") == subtag_name:
            return item.get("value", default)
    return default


def find_subtag_value_any(container, subtag_name, default="N/A"):
    """More robust subtag extraction.

    Compatible with:
    - list[{'subtag','value'}]
    - dict with {'subtags': [...]}
    - single dict with {'subtag': 'XXX', 'value': '...'}
    - direct key->value in some implementations
    """
    if container is None:
        return default

    if isinstance(container, list):
        return find_subtag_value(container, subtag_name, default)

    if isinstance(container, dict):
        if container.get("subtag") == subtag_name:
            return container.get("value", default)
        if isinstance(container.get("subtags"), list):
            return find_subtag_value(container.get("subtags"), subtag_name, default)
        if subtag_name in container:
            return container.get(subtag_name, default)

    return default


def coalesce(*vals, default="N/A"):
    """Return the first non-empty/non-N/A value."""
    for v in vals:
        if v is None:
            continue
        sv = str(v).strip()
        if sv == "" or sv == "N/A" or sv == "nan":
            continue
        return v
    return default


# -------------------- HTTP Session + Cache + Rate Limiting --------------------

_thread_local = threading.local()
_cache_lock = threading.Lock()
_record_cache = {}  # accession -> record(dict)

_rate_lock = threading.Lock()
_last_request_ts = 0.0


def _throttle_if_needed():
    """Apply rate limiting to avoid overwhelming the API."""
    global _last_request_ts
    if REQUEST_INTERVAL_SECONDS <= 0:
        return
    with _rate_lock:
        now = time.monotonic()
        wait_s = REQUEST_INTERVAL_SECONDS - (now - _last_request_ts)
        if wait_s > 0:
            time.sleep(wait_s)
            _last_request_ts = time.monotonic()
        else:
            _last_request_ts = now


def create_session_with_retries(retries: int = RETRY_COUNT) -> requests.Session:
    """Create a requests.Session with retry mechanism."""
    session = requests.Session()
    retry_strategy = Retry(
        total=retries,
        backoff_factor=0.3,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def get_thread_session() -> requests.Session:
    """One Session per thread to avoid potential issues from sharing Sessions across threads."""
    if not hasattr(_thread_local, "session"):
        _thread_local.session = create_session_with_retries()
    return _thread_local.session


def fetch_json(url: str, session: requests.Session, timeout: int = 30):
    """Fetch and parse JSON from URL."""
    _throttle_if_needed()
    resp = session.get(url, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def fetch_records_by_inchikey(inchi_key: str, session: requests.Session):
    """Get record list from MassBank API by InChIKey."""
    if pd.isna(inchi_key) or str(inchi_key).strip() == "":
        tqdm.write("Warning: InChIKey is empty, skipping query.")
        return None

    url = f"{API_RECORDS_URL}?inchi_key={inchi_key}"
    try:
        data = fetch_json(url, session)
        if isinstance(data, dict) and "records" in data and isinstance(data["records"], list):
            return data["records"]
        return data
    except requests.exceptions.RequestException as e:
        tqdm.write(f"Warning: Network error when querying InChIKey '{inchi_key}': {e}")
        return None
    except ValueError:
        tqdm.write(f"Warning: Cannot parse JSON response for InChIKey '{inchi_key}'. Possibly no records found.")
        return None


def fetch_record_by_accession(accession: str, session: requests.Session):
    """Fetch single record details using /records/{accession} endpoint (with caching)."""
    if not accession or str(accession).strip() == "":
        return None

    accession = str(accession).strip()

    with _cache_lock:
        if accession in _record_cache:
            return _record_cache[accession]

    url = API_RECORD_URL_TMPL.format(accession=accession)
    try:
        rec = fetch_json(url, session)
        with _cache_lock:
            _record_cache[accession] = rec
        return rec
    except requests.exceptions.RequestException as e:
        tqdm.write(f"Warning: Network error when fetching accession '{accession}' details: {e}")
        return None
    except ValueError:
        tqdm.write(f"Warning: Cannot parse JSON response for accession '{accession}'.")
        return None


# -------------------- Field Extraction: Ion Mode / LC / Collision Energy --------------------

def get_lc_params_from_record(record: dict) -> dict:
    """Extract LC/chromatography related parameters: Column Name, Flow Gradient, Flow Rate, Retention Time."""
    chrom = extract_nested_value(record, ["acquisition", "chromatography"], [])

    column_name = coalesce(
        find_subtag_value_any(chrom, "COLUMN_NAME"),
        find_subtag_value_any(chrom, "COLUMN"),
        find_subtag_value_any(chrom, "COLUMN_DESCRIPTION"),
        default="N/A",
    )

    flow_gradient = coalesce(
        find_subtag_value_any(chrom, "FLOW_GRADIENT"),
        find_subtag_value_any(chrom, "GRADIENT"),
        find_subtag_value_any(chrom, "FLOW_GRADIENT_PROFILE"),
        default="N/A",
    )

    flow_rate = coalesce(
        find_subtag_value_any(chrom, "FLOW_RATE"),
        find_subtag_value_any(chrom, "FLOW"),
        default="N/A",
    )

    retention_time = coalesce(
        find_subtag_value_any(chrom, "RETENTION_TIME"),
        find_subtag_value_any(chrom, "RETENTION"),
        find_subtag_value_any(chrom, "RT"),
        default="N/A",
    )

    # Fallback for records where data might be directly in dict->subtags
    if column_name == "N/A" and isinstance(chrom, dict):
        column_name = find_subtag_value_any(chrom.get("subtags"), "COLUMN_NAME")
    if flow_gradient == "N/A" and isinstance(chrom, dict):
        flow_gradient = find_subtag_value_any(chrom.get("subtags"), "FLOW_GRADIENT")
    if flow_rate == "N/A" and isinstance(chrom, dict):
        flow_rate = find_subtag_value_any(chrom.get("subtags"), "FLOW_RATE")
    if retention_time == "N/A" and isinstance(chrom, dict):
        retention_time = find_subtag_value_any(chrom.get("subtags"), "RETENTION_TIME")

    # Additional fallback for common aliases
    if column_name == "N/A":
        column_name = find_subtag_value_any(chrom, "COLUMN")
    if flow_gradient == "N/A":
        flow_gradient = find_subtag_value_any(chrom, "GRADIENT")
    if flow_rate == "N/A":
        flow_rate = find_subtag_value_any(chrom, "FLOW")
    if retention_time == "N/A":
        retention_time = find_subtag_value_any(chrom, "RETENTION")
    if retention_time == "N/A":
        retention_time = find_subtag_value_any(chrom, "RT")

    return {
        "Column Name": column_name,
        "Flow Gradient": flow_gradient,
        "Flow Rate": flow_rate,
        "Retention Time": retention_time,
    }


def get_ion_mode_from_record(record: dict) -> str:
    """Extract Ion Mode. Priority: acquisition.mass_spectrometry.subtags: ION_MODE."""
    ms_acq = extract_nested_value(record, ["acquisition", "mass_spectrometry"], {})
    ion_mode = find_subtag_value_any(ms_acq, "ION_MODE")

    if ion_mode == "N/A":
        for path in (
                ["ion_mode"],
                ["mass_spectrometry", "ion_mode"],
                ["acquisition", "mass_spectrometry", "ion_mode"],
        ):
            v = extract_nested_value(record, path, "N/A")
            if v != "N/A":
                ion_mode = v
                break

    if ion_mode == "N/A":
        ms_root = record.get("mass_spectrometry", {})
        ion_mode = find_subtag_value_any(ms_root, "ION_MODE")
        if ion_mode == "N/A" and isinstance(ms_root, dict):
            ion_mode = find_subtag_value_any(ms_root.get("subtags"), "ION_MODE")

    return ion_mode


def extract_collision_energy(record: dict) -> str:
    """Extract Collision Energy from record.

    MassBank records typically have it under AC$MASS_SPECTROMETRY's COLLISION_ENERGY.
    Different record/parser implementations may place it in:
    - acquisition.mass_spectrometry.subtags
    - mass_spectrometry.subtags
    - mass_spectrometry.focused_ion (less common)
    - collision_energy (some implementations provide it directly)

    Returns string; returns 'N/A' if not found.
    """
    candidates = [
        "COLLISION_ENERGY",
        "COLLISION ENERGY",
        "COLLISION_ENERGY_EV",
        "COLLISION_ENERGY_V",
        "COLLISIONENERGY",
    ]

    # 1) acquisition.mass_spectrometry.subtags
    ms_acq = extract_nested_value(record, ["acquisition", "mass_spectrometry"], None)
    for name in candidates:
        v = find_subtag_value_any(ms_acq, name)
        if v != "N/A":
            return v
        if isinstance(ms_acq, dict):
            v2 = find_subtag_value_any(ms_acq.get("subtags"), name)
            if v2 != "N/A":
                return v2

    # 2) record.mass_spectrometry.subtags
    ms_root = record.get("mass_spectrometry", None)
    for name in candidates:
        v = find_subtag_value_any(ms_root, name)
        if v != "N/A":
            return v
        if isinstance(ms_root, dict):
            v2 = find_subtag_value_any(ms_root.get("subtags"), name)
            if v2 != "N/A":
                return v2

    # 3) focused_ion
    focused_ion = extract_nested_value(record, ["mass_spectrometry", "focused_ion"], [])
    for name in candidates:
        v = find_subtag_value_any(focused_ion, name)
        if v != "N/A":
            return v

    # 4) Direct fields
    v_direct = coalesce(
        extract_nested_value(record, ["collision_energy"], None),
        extract_nested_value(record, ["acquisition", "collision_energy"], None),
        extract_nested_value(record, ["acquisition", "mass_spectrometry", "collision_energy"], None),
        extract_nested_value(record, ["mass_spectrometry", "collision_energy"], None),
        default="N/A",
    )
    return v_direct


def maybe_enrich_for_collision_energy(record: dict, session: requests.Session) -> dict:
    """Call /records/{accession} to fetch details and extract Collision Energy when needed.

    - If strategy is 'always': always fetch
    - If strategy is 'if_missing': only fetch when current record's Collision Energy is N/A

    Returns the "as complete as possible" record (which may still be the original record).
    """
    if not isinstance(record, dict):
        return record

    accession = extract_nested_value(record, ["accession"], None)
    if not accession:
        return record

    if COLLISION_ENERGY_FETCH_MODE == "always":
        detail = fetch_record_by_accession(accession, session)
        return detail if isinstance(detail, dict) else record

    # if_missing
    ce = extract_collision_energy(record)
    if ce != "N/A":
        return record

    detail = fetch_record_by_accession(accession, session)
    if isinstance(detail, dict):
        return detail
    return record


# -------------------- Mass Spectrum Peak Feature Calculation --------------------

def calculate_peak_features(peak_df: pd.DataFrame) -> pd.DataFrame:
    """Calculate feature values from mass spectrum peak data."""
    if peak_df.empty:
        feature_cols = [
            "base peak mass",
            "base peak mass proximity",
            "MaxM",
            "MaxMP",
            "MinM",
            "PN",
            "ID",
            "MM",
            "MSD",
            "IM",
            "ISD",
            "Mass Density",
            "Most Frequent PPMD",
            "Mean PPMD",
        ]
        return pd.DataFrame({col: ["N/A"] for col in feature_cols})

    df = peak_df.rename(columns={"m/z": "Mass", "int.": "Intensity", "rel.int.": "Rel_Intensity"})
    df["Mass"] = pd.to_numeric(df["Mass"], errors="coerce")
    df["Intensity"] = pd.to_numeric(df["Intensity"], errors="coerce")
    df["Rel_Intensity"] = pd.to_numeric(df["Rel_Intensity"], errors="coerce")
    df.dropna(inplace=True)
    if df.empty:
        return calculate_peak_features(pd.DataFrame())

    float_FF1 = df["Mass"].tolist()
    float_FF3 = df["Rel_Intensity"].tolist()

    PN = len(df)
    ID = (max(float_FF3) if float_FF3 else 0) / PN if PN > 0 else 0

    max_FF3_val = max(float_FF3)
    bp_index = float_FF3.index(max_FF3_val)
    max_FF3_FF1 = float_FF1[bp_index]

    if PN <= 1:
        bpp, max_mp = 0, 0
    else:
        distances = [abs(float_FF1[i] - max_FF3_FF1) for i in range(PN) if i != bp_index]
        bpp = min(distances) if distances else 0

        max_FF1 = max(float_FF1)
        max_index_mz = float_FF1.index(max_FF1)
        distances_max = [abs(float_FF1[i] - max_FF1) for i in range(PN) if i != max_index_mz]
        max_mp = min(distances_max) if distances_max else 0

    max_FF1 = max(float_FF1)
    min_FF1 = min(float_FF1)

    FF1_average = sum(float_FF1) / len(float_FF1) if float_FF1 else 0
    FF1_bzc = np.std(float_FF1) if float_FF1 else 0

    FF2_average = sum(float_FF3) / len(float_FF3) if float_FF3 else 0
    FF2_bzc = np.std(float_FF3) if float_FF3 else 0

    mass_density = max_FF1 / PN if PN > 0 else 0

    if PN <= 1:
        most_frequent_ppmd, mean_ppmd = 0, 0
    else:
        ppmds = [abs(float_FF1[i] - float_FF1[j]) for i in range(PN) for j in range(i + 1, PN)]
        if ppmds:
            mean_ppmd = sum(ppmds) / len(ppmds)
            rounded_ppmds = [round(diff, 2) for diff in ppmds]
            freq_counter = Counter(rounded_ppmds)
            most_frequent_ppmd = freq_counter.most_common(1)[0][0] if freq_counter else 0
        else:
            most_frequent_ppmd, mean_ppmd = 0, 0

    return pd.DataFrame(
        {
            "base peak mass": [max_FF3_FF1],
            "base peak mass proximity": [bpp],
            "MaxM": [max_FF1],
            "MaxMP": [max_mp],
            "MinM": [min_FF1],
            "PN": [PN],
            "ID": [ID],
            "MM": [FF1_average],
            "MSD": [FF1_bzc],
            "IM": [FF2_average],
            "ISD": [FF2_bzc],
            "Mass Density": [mass_density],
            "Most Frequent PPMD": [most_frequent_ppmd],
            "Mean PPMD": [mean_ppmd],
        }
    )


def build_peak_df_from_record(record: dict) -> pd.DataFrame:
    """Build peak_df from record.peak.peak.values (compatible with different structures)."""
    peak_data = extract_nested_value(record, ["peak", "peak", "values"], [])
    if not peak_data:
        peak_data = extract_nested_value(record, ["peak", "values"], [])
    if not peak_data:
        return pd.DataFrame()

    df = pd.DataFrame(peak_data)

    # Column name compatibility
    if "mz" in df.columns:
        df = df.rename(columns={"mz": "m/z"})
    if "intensity" in df.columns:
        df = df.rename(columns={"intensity": "int."})
    if "rel" in df.columns:
        df = df.rename(columns={"rel": "rel.int."})
    if "relative" in df.columns and "rel.int." not in df.columns:
        df = df.rename(columns={"relative": "rel.int."})

    # If rel.int. is missing but int. exists, create empty column
    if "rel.int." not in df.columns and "int." in df.columns:
        df["rel.int."] = np.nan

    if all(c in df.columns for c in ["m/z", "int.", "rel.int."]):
        return df[["m/z", "int.", "rel.int."]].copy()
    return pd.DataFrame()


# -------------------- Row Processing Logic --------------------

def build_error_df(error_reason: str = "NOT_FOUND") -> pd.DataFrame:
    """Build error DataFrame with consistent columns."""
    api_columns = [
        "accession",
        "title",
        "classes",
        "mass",
        "Ion Mode",
        "Column Name",
        "Flow Gradient",
        "Flow Rate",
        "Retention Time",
        "Base Peak",
        "Precursor M/z",
        "Precursor Type",
        "retention time",
        "Collision Energy",
        "base peak mass",
        "base peak mass proximity",
        "MaxM",
        "MaxMP",
        "MinM",
        "PN",
        "ID",
        "MM",
        "MSD",
        "IM",
        "ISD",
        "Mass Density",
        "Most Frequent PPMD",
        "Mean PPMD",
    ]
    return pd.DataFrame({col: [error_reason] for col in api_columns})


def process_row(original_row_tuple):
    """Process a single row: query API, parse fields, calculate peak features, return DataFrame (may be multiple rows)."""
    idx, original_row = original_row_tuple
    inchi_key = original_row["InChIKey"]
    original_row_df = original_row.to_frame().T

    session = get_thread_session()
    records = fetch_records_by_inchikey(inchi_key, session)

    if not records:
        return pd.concat([original_row_df.reset_index(drop=True), build_error_df("NO_RECORDS_FOUND")], axis=1)

    # Some responses may wrap records in a dict; handle gracefully
    if isinstance(records, dict):
        records = [records]
    if not isinstance(records, list):
        return pd.concat([original_row_df.reset_index(drop=True), build_error_df("INVALID_RESPONSE")], axis=1)

    all_results_for_row = []
    for record in records:
        if not isinstance(record, dict):
            continue

        # Enrich with details if needed (for Collision Energy)
        record = maybe_enrich_for_collision_energy(record, session)

        classes_value = extract_nested_value(record, ["compound", "classes"], [])
        classes_str = ", ".join(classes_value) if isinstance(classes_value, list) else "N/A"

        ion_mode = get_ion_mode_from_record(record)
        lc_params = get_lc_params_from_record(record)

        focused_ion = extract_nested_value(record, ["mass_spectrometry", "focused_ion"], [])
        base_peak = find_subtag_value(focused_ion, "BASE_PEAK")
        precursor_mz = find_subtag_value(focused_ion, "PRECURSOR_M/Z")
        precursor_type = find_subtag_value(focused_ion, "PRECURSOR_TYPE")

        ce = extract_collision_energy(record)

        metadata = {
            "accession": extract_nested_value(record, ["accession"]),
            "title": extract_nested_value(record, ["title"]),
            "classes": classes_str,
            "mass": extract_nested_value(record, ["compound", "mass"]),
            "Ion Mode": ion_mode,
            **lc_params,
            "Base Peak": base_peak,
            "Precursor M/z": precursor_mz,
            "Precursor Type": precursor_type,
            "retention time": lc_params.get("Retention Time", "N/A"),
            "Collision Energy": ce,
        }

        metadata_df = pd.DataFrame([metadata])

        peak_df = build_peak_df_from_record(record)
        feature_df = calculate_peak_features(peak_df)

        full_result = pd.concat([original_row_df.reset_index(drop=True), metadata_df, feature_df], axis=1)
        all_results_for_row.append(full_result)

    if not all_results_for_row:
        return pd.concat([original_row_df.reset_index(drop=True), build_error_df("NO_VALID_RECORDS")], axis=1)

    return pd.concat(all_results_for_row, ignore_index=True)


# -------------------- Output Writing --------------------

def save_batch_to_csv(batch, filename: str):
    """Save a batch of results to CSV (append mode)."""
    if not batch:
        return
    final_df_batch = pd.concat(batch, ignore_index=True)
    os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)
    header = not os.path.exists(filename)
    final_df_batch.to_csv(filename, mode="a", header=header, index=False)


# -------------------- Main Function --------------------

def main():
    try:
        input_df = pd.read_excel(INPUT_FILE)
    except FileNotFoundError:
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        return

    if "InChIKey" not in input_df.columns:
        print("Error: Column 'InChIKey' not found in input file.")
        return

    processed_keys = get_processed_inchikeys(OUTPUT_FILE)
    df_to_process = input_df[~input_df["InChIKey"].isin(processed_keys)].copy()

    if df_to_process.empty:
        print(f"All InChIKeys in '{INPUT_FILE}' have already been processed.")
        return

    print(f"Found {len(input_df)} total rows, {len(df_to_process)} rows need processing.")

    results_batch = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_row = {executor.submit(process_row, row): row for row in df_to_process.iterrows()}

        for future in tqdm(concurrent.futures.as_completed(future_to_row), total=len(df_to_process), desc="Processing rows"):
            try:
                result_df = future.result()
                if result_df is not None:
                    results_batch.append(result_df)

                if len(results_batch) >= BATCH_SIZE:
                    save_batch_to_csv(results_batch, OUTPUT_FILE)
                    results_batch = []

            except Exception as exc:
                tqdm.write(f"Exception while processing a row: {exc}")

    save_batch_to_csv(results_batch, OUTPUT_FILE)
    print(f"\nProcessing complete! Results saved to '{OUTPUT_FILE}'.")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

data = pd.read_csv('data.csv')

X = data.drop(['label'], axis=1)
y = data['label']

unique_names = X['name'].unique()

train_names, test_names = train_test_split(
    unique_names, 
    test_size=0.2, 
    random_state=42
)

train_mask = X['name'].isin(train_names)
test_mask = X['name'].isin(test_names)

X_train_raw = X[train_mask].copy()
X_test_raw = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

use_smiles = False
inchikey_to_smiles = {}

if use_smiles:
    X_train_raw['SMILES'] = X_train_raw['name'].map(inchikey_to_smiles)
    X_test_raw['SMILES'] = X_test_raw['name'].map(inchikey_to_smiles)

train_names_saved = X_train_raw['name'].copy()
test_names_saved = X_test_raw['name'].copy()

if use_smiles:
    train_smiles_saved = X_train_raw['SMILES'].copy()
    test_smiles_saved = X_test_raw['SMILES'].copy()
    X_train = X_train_raw.drop(columns=['name', 'SMILES'])
    X_test = X_test_raw.drop(columns=['name', 'SMILES'])
else:
    X_train = X_train_raw.drop(columns=['name'])
    X_test = X_test_raw.drop(columns=['name'])

continuous_features = X_train.select_dtypes(include=['float64', 'int64']).columns

X_train = X_train[continuous_features]
X_test = X_test[continuous_features]

for column in X_train.columns:
    if X_train[column].dtype in ['float64', 'int64']:
        median_value = X_train[column].median() if X_train[column].notna().any() else 0
        X_train[column].fillna(median_value, inplace=True)
        if column in X_test.columns:
            X_test[column].fillna(median_value, inplace=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_before_smote = X_train_scaled.copy()
y_train_before_smote = y_train.copy()

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_before_smote, y_train_before_smote)

X_train = pd.DataFrame(
    X_train_resampled, 
    columns=continuous_features
)

X_test = pd.DataFrame(X_test_scaled, columns=continuous_features, index=X_test.index)

y_train = y_train_resampled